In [2]:
%cd /home/luke-padmore/Source/flow-matching-mnist

/home/luke-padmore/Source/flow-matching-mnist


In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

import mlflow
import mlflow.pytorch
import pandas as pd
import torch
from models.ode_solvers import (
    create_samples,
    get_ode_solver_from_name,
    make_vf_uncond,
)
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid
from utils.FID.fid_evaluation import evaluate_fid_with_registered_backbone
from utils.data_modules import MNISTDataModule

# For system metrics
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

batch_size = 64
mnist_dm = MNISTDataModule(
    batch_size=batch_size,
    data_path="data",
    num_workers=4,
    transform="default",
)
trainloader = mnist_dm.train_dataloader()


In [4]:
run_id = "f37b6e2327874ed5a9ca4620e02b0361"
model = mlflow.pytorch.load_model(f"runs:/{run_id}/UNet")
print(model)

/home/luke-padmore/miniconda3/envs/ml/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)

UNet(
  (encoder): Encoder(
    (initial_conv): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): ConvDownblock(
        (conv): Sequential(
          (0): GroupNorm(8, 72, eps=1e-05, affine=True)
          (1): SiLU()
          (2): Conv2d(72, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (3): GroupNorm(8, 64, eps=1e-05, affine=True)
          (4): SiLU()
          (5): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (down): Sequential(
          (0): GroupNorm(8, 64, eps=1e-05, affine=True)
          (1): SiLU()
          (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        )
        (time_emb_mlp): Linear(in_features=32, out_features=8, bias=True)
      )
      (1): ConvDownblock(
        (conv): Sequential(
          (0): GroupNorm(8, 72, eps=1e-05, affine=True)
          (1): SiLU()
          (2): Conv2d(72, 1

In [7]:
from models.ode_solvers import euler_solver 
from utils.mlflow_tracking_utils import get_run_param
ode_solver = euler_solver
image_shape = (1,32,32)
ode_steps = 50
batch_size = int(get_run_param(run_id,"batch_size"))
 
f = make_vf_uncond(model)

sample_fn = partial(
    create_samples,
    image_shape= image_shape,
    ode_solver = ode_solver,
    f=f,
    n_steps=ode_steps,
    seed=None,
    device=device,
)
sample_fn

functools.partial(<function create_samples at 0x7572913ddbc0>, image_shape=(1, 32, 32), ode_solver=<function euler_solver at 0x7572913dd800>, f=<function make_vf_uncond.<locals>.f at 0x75737c8385e0>, n_steps=50, seed=None, device=device(type='cuda', index=0))

In [10]:
from torchvision.utils import save_image
samples = create_samples(
    n_images=64,                  # 8x8 grid
    image_shape=[1, 32, 32],
    ode_solver=ode_solver,
    f=f,
    n_steps=50,
    seed=42,
    device=device,
)

# unnormalize from [-1,1] -> [0,1]
samples = ((samples + 1) / 2).clamp(0, 1)

grid = make_grid(samples, nrow=8, padding=2)
out_dir = Path("notebooks/fid_exports")
out_dir.mkdir(parents=True, exist_ok=True)
save_image(grid, out_dir / "sample_grid_blog.png")


In [ ]:
from functools import partial
from pathlib import Path

import mlflow
import mlflow.pytorch
from torchvision.utils import make_grid

from models.ode_solvers import (
    create_samples,
    make_vf_uncond,
)
from utils.data_modules import MNISTDataModule

run_id = "dfcf16135aae46128df6fbbadb80ec00"
model = mlflow.pytorch.load_model(f"runs:/{run_id}/UNet")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# assumes run_id, model, device already exist in notebook
val_loader = MNISTDataModule(
    batch_size=128,
    data_path="data",
    transform="default",
).val_dataloader()

sample_samplers = ["euler_solver", "rk2_solver", "rk4_solver"]
sample_step_counts = [25, 50, 100]
image_shape = [1, 32, 32]
eval_batch_size = 32
n_fid_samples = 3200

results = []
mlflow.set_experiment('flow-matching-debugging')
with mlflow.start_run(run_name=f"fid_and_samples::{run_id[:8]}"):
    mlflow.log_param("generator_run_id", run_id)
    mlflow.log_param("n_fid_samples", n_fid_samples)
    mlflow.log_param("image_shape", ",".join(map(str, image_shape)))

    f = make_vf_uncond(model)

    for sampler in sample_samplers:
        ode_solver = get_ode_solver_from_name(sampler)

        for ode_steps in sample_step_counts:
            print(f"Running {sampler} with {ode_steps} steps")

            sample_fn = partial(
                create_samples,
                image_shape=image_shape,
                ode_solver=ode_solver,
                f=f,
                n_steps=ode_steps,
                seed=None,
                device=device,
            )

            fid, gen_embs, real_embs = evaluate_fid_with_registered_backbone(
                sample_fn=sample_fn,
                device=device,
                n_samples=n_fid_samples,
                batch_size=eval_batch_size,
                real_loader=val_loader,
                show_progress=True,
            )

            results.append(
                {
                    "sampler": sampler,
                    "ode_steps": ode_steps,
                    "fid": float(fid),
                    "gen_embs_shape": tuple(gen_embs.shape),
                    "real_embs_shape": None if real_embs is None else tuple(real_embs.shape),
                }
            )

            mlflow.log_metric(f"fid_{sampler}_{ode_steps}", float(fid))

            # log one grid artifact per sampler/step
            samples = create_samples(
                n_images=64,
                image_shape=image_shape,
                ode_solver=ode_solver,
                f=f,
                n_steps=ode_steps,
                seed=42,
                device=device,
            )
            samples = ((samples + 1) / 2).clamp(0, 1)
            grid = make_grid(samples, nrow=8, padding=2)
            grid_img = to_pil_image(grid.cpu())

            mlflow.log_image(
                grid_img,
                artifact_file=f"sample_grids/{sampler}_{ode_steps:03d}_steps.png",
            )

    results_df = pd.DataFrame(results).sort_values(["fid", "ode_steps"])
    csv_path = Path("/tmp/fid_sampler_sweep.csv")
    results_df.to_csv(csv_path, index=False)
    mlflow.log_artifact(str(csv_path), artifact_path="tables")
    
results_df
